# Machine Learning Lab - Classification & Model Comparison Analysis

This notebook evaluates **Naïve Bayes** (Gaussian, Multinomial, Bernoulli) and **K-Nearest Neighbors (KNN)** algorithms on the Spambase dataset.
It includes:
- Step 1–3: Dataset loading, explicit missing value checks, and feature scaling (`MinMaxScaler`)
- Step 4: Exploratory Data Analysis (Class distribution, correlation heatmap, feature distributions, boxplots)
- Step 5–11: Performance evaluation, `GridSearchCV`, `RandomizedSearchCV`, `KDTree` vs `BallTree`, 5-Fold Cross Validation, and Experimental Time Complexity Analysis
- Step 12: Complete suite of **14 required evaluation plots** and confusion matrix visualizations.

In [ ]:
import time
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, KFold
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve,
    average_precision_score, ConfusionMatrixDisplay
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
OUTPUT_DIR = "output_plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Step 1: Load Dataset
dataset_path = "spambase_csv.csv"
if not os.path.exists(dataset_path):
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data'
    df = pd.read_csv(url, header=None)
    df.rename(columns={57: 'class'}, inplace=True)
    df.to_csv(dataset_path, index=False)
else:
    df = pd.read_csv(dataset_path)

df.columns = df.columns.str.strip()
target_col = 'class' if 'class' in df.columns else df.columns[-1]

# Step 2: Handle Missing Values Explicitly
missing_count = df.isnull().sum().sum()
print(f"Explicit Missing Values Count: {missing_count}")
if missing_count > 0:
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].fillna(df[col].median())
            else:
                df[col] = df[col].fillna(df[col].mode()[0])
    print("Missing values successfully handled!")

for col in df.select_dtypes(include=['object']).columns:
    df[col] = LabelEncoder().fit_transform(df[col])

X = df.drop(target_col, axis=1)
y = df[target_col]

# Step 3: Normalize Features
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Step 5: Split Data into Train and Test Sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f"Dataset Loaded & Scaled: {X.shape[0]} samples, {X.shape[1]} features")

### Table 1: Naïve Bayes Comparison

In [ ]:
nb_models = {
    'Gaussian NB': GaussianNB(),
    'Multinomial NB': MultinomialNB(),
    'Bernoulli NB': BernoulliNB()
}

nb_metrics = {}
nb_preds = {}
nb_probs = {}

for name, model in nb_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else pred
    
    nb_preds[name] = pred
    nb_probs[name] = prob
    
    nb_metrics[name] = [
        round(accuracy_score(y_test, pred), 4),
        round(precision_score(y_test, pred, zero_division=0), 4),
        round(recall_score(y_test, pred, zero_division=0), 4),
        round(f1_score(y_test, pred, zero_division=0), 4),
        round(roc_auc_score(y_test, prob), 4)
    ]

table1 = pd.DataFrame(nb_metrics, index=['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC'])
table1.index.name = 'Metric'
table1.reset_index(inplace=True)

print("--- Table 1: Naïve Bayes Comparison ---")
print(table1.to_string(index=False))

### Table 2: KNN Comparison

In [ ]:
k_values = [1, 3, 5, 7, 9, 11]
knn_rows = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    pred = knn.predict(X_test)
    
    knn_rows.append({
        'k': k,
        'Accuracy': round(accuracy_score(y_test, pred), 4),
        'Precision': round(precision_score(y_test, pred, zero_division=0), 4),
        'Recall': round(recall_score(y_test, pred, zero_division=0), 4),
        'F1': round(f1_score(y_test, pred, zero_division=0), 4)
    })

table2 = pd.DataFrame(knn_rows)
print("--- Table 2: KNN Comparison ---")
print(table2.to_string(index=False))

### Table 3: Grid Search vs Randomized Search

In [ ]:
param_grid = {
    'n_neighbors': [1, 3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}

t0 = time.time()
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
grid_time = time.time() - t0

t0 = time.time()
rand_search = RandomizedSearchCV(KNeighborsClassifier(), param_grid, n_iter=15, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
rand_search.fit(X_train, y_train)
rand_time = time.time() - t0

table3 = pd.DataFrame({
    'Parameter': ['Best k', 'Metric', 'Weights', 'Algorithm', 'CV Accuracy', 'Execution Time'],
    'GridSearchCV': [
        grid_search.best_params_['n_neighbors'],
        grid_search.best_params_['metric'],
        grid_search.best_params_['weights'],
        grid_search.best_params_['algorithm'],
        round(grid_search.best_score_, 4),
        f"{grid_time:.4f} s"
    ],
    'RandomizedSearchCV': [
        rand_search.best_params_['n_neighbors'],
        rand_search.best_params_['metric'],
        rand_search.best_params_['weights'],
        rand_search.best_params_['algorithm'],
        round(rand_search.best_score_, 4),
        f"{rand_time:.4f} s"
    ]
})

print("--- Table 3: Grid Search vs Randomized Search ---")
print(table3.to_string(index=False))

best_knn = grid_search.best_estimator_
best_knn_pred = best_knn.predict(X_test)
best_knn_prob = best_knn.predict_proba(X_test)[:, 1]

all_preds = {**nb_preds, 'Best KNN': best_knn_pred}
all_probs = {**nb_probs, 'Best KNN': best_knn_prob}

### Table 4: KDTree vs BallTree

In [ ]:
kd_clf = KNeighborsClassifier(algorithm='kd_tree')
t0 = time.perf_counter()
kd_clf.fit(X_train, y_train)
kd_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
kd_pred = kd_clf.predict(X_test)
kd_pred_time = time.perf_counter() - t0
kd_acc = accuracy_score(y_test, kd_pred)

ball_clf = KNeighborsClassifier(algorithm='ball_tree')
t0 = time.perf_counter()
ball_clf.fit(X_train, y_train)
ball_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
ball_pred = ball_clf.predict(X_test)
ball_pred_time = time.perf_counter() - t0
ball_acc = accuracy_score(y_test, ball_pred)

table4 = pd.DataFrame({
    'Metric': ['Accuracy', 'Training Time', 'Prediction Time'],
    'KDTree': [round(kd_acc, 4), f"{kd_train_time:.6f} s", f"{kd_pred_time:.6f} s"],
    'BallTree': [round(ball_acc, 4), f"{ball_train_time:.6f} s", f"{ball_pred_time:.6f} s"]
})

print("--- Table 4: KDTree vs BallTree ---")
print(table4.to_string(index=False))

### Table 5: Cross Validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
best_nb = GaussianNB()
nb_scores = cross_val_score(best_nb, X_scaled, y, cv=kf)
knn_scores = cross_val_score(best_knn, X_scaled, y, cv=kf)

table5_rows = []
for idx in range(5):
    table5_rows.append({
        'Fold': str(idx + 1),
        'Naïve Bayes': round(nb_scores[idx], 4),
        'Best KNN': round(knn_scores[idx], 4)
    })

table5_rows.append({
    'Fold': 'Average',
    'Naïve Bayes': round(nb_scores.mean(), 4),
    'Best KNN': round(knn_scores.mean(), 4)
})

table5 = pd.DataFrame(table5_rows)
print("--- Table 5: Cross Validation ---")
print(table5.to_string(index=False))

### Table 6: Theoretical Time Complexity

In [ ]:
table6 = pd.DataFrame({
    'Algorithm': ['Naïve Bayes', 'KNN (Brute)', 'KDTree', 'BallTree'],
    'Training': ['O(nd)', 'O(1)', 'O(n log n)', 'O(n log n)'],
    'Prediction': ['O(d)', 'O(nd)', 'O(log n) avg', 'O(log n) avg']
})

print("--- Table 6: Theoretical Time Complexity ---")
print(table6.to_string(index=False))

### Table 7: Experimental Time Analysis

In [ ]:
exp_models = {
    'Gaussian NB': GaussianNB(),
    'Multinomial NB': MultinomialNB(),
    'Bernoulli NB': BernoulliNB(),
    'Best KNN': best_knn
}

exp_rows = []
exp_train_times = {}
exp_pred_times = {}

for name, model in exp_models.items():
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    tr_time = time.perf_counter() - t0
    
    t0 = time.perf_counter()
    model.predict(X_test)
    pr_time = time.perf_counter() - t0
    
    exp_train_times[name] = tr_time
    exp_pred_times[name] = pr_time
    
    exp_rows.append({
        'Algorithm': name,
        'Training(s)': f"{tr_time:.6f}",
        'Prediction(s)': f"{pr_time:.6f}"
    })

table7 = pd.DataFrame(exp_rows)
print("--- Table 7: Experimental Time Analysis ---")
print(table7.to_string(index=False))

## Required Plots (Plots 1 to 14)

In [ ]:
# Plot 1: Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x=y, palette='viridis', hue=y, legend=False)
plt.title("Plot 1: Class Distribution (0: Non-Spam, 1: Spam)", fontsize=12, fontweight='bold')
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "1_class_distribution.png"))
plt.show()

In [ ]:
# Plot 2: Correlation Heatmap
plt.figure(figsize=(10, 8))
corrs = df.corr()[target_col].abs().sort_values(ascending=False)
top_corr_cols = corrs.index[:12]
sns.heatmap(df[top_corr_cols].corr(), annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title("Plot 2: Correlation Heatmap (Top Correlated Features)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "2_correlation_heatmap.png"))
plt.show()

In [ ]:
# Plot 3: Histograms of Important Features
top_features = corrs.index[1:7]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for idx, col in enumerate(top_features):
    sns.histplot(df, x=col, hue=target_col, kde=True, ax=axes[idx], palette='Set1', bins=25)
    axes[idx].set_title(f"Dist of {col}")
plt.suptitle("Plot 3: Histograms of Important Features", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "3_feature_histograms.png"))
plt.show()

In [ ]:
# Plot 4: Boxplots
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for idx, col in enumerate(top_features):
    sns.boxplot(data=df, x=target_col, y=col, ax=axes[idx], palette='Set2', hue=target_col, legend=False)
    axes[idx].set_title(f"Boxplot of {col} by Class")
plt.suptitle("Plot 4: Feature Boxplots", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "4_feature_boxplots.png"))
plt.show()

In [ ]:
# Plot 5: Confusion Matrix for Each Model
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
axes = axes.flatten()
for idx, (name, pred) in enumerate(all_preds.items()):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Spam', 'Spam'])
    disp.plot(ax=axes[idx], cmap='Blues', colorbar=False)
    axes[idx].set_title(f"CM: {name}", fontweight='bold')
plt.suptitle("Plot 5: Confusion Matrix for Each Model", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "5_confusion_matrices.png"))
plt.show()

In [ ]:
# Plot 6: ROC Curves
plt.figure(figsize=(8, 6))
for name, prob in all_probs.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.4f})", linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Chance (AUC = 0.5000)')
plt.title("Plot 6: ROC Curves Comparison", fontsize=12, fontweight='bold')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "6_roc_curves.png"))
plt.show()

In [ ]:
# Plot 7: Precision-Recall Curves
plt.figure(figsize=(8, 6))
for name, prob in all_probs.items():
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap_val = average_precision_score(y_test, prob)
    plt.plot(rec, prec, label=f"{name} (AP = {ap_val:.4f})", linewidth=2)
plt.title("Plot 7: Precision-Recall Curves Comparison", fontsize=12, fontweight='bold')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(loc='lower left')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "7_precision_recall_curves.png"))
plt.show()

In [ ]:
# Plot 8: Accuracy vs k
plt.figure(figsize=(8, 5))
plt.plot(table2['k'], table2['Accuracy'], marker='o', color='purple', linewidth=2, markersize=8)
plt.title("Plot 8: KNN Accuracy vs k", fontsize=12, fontweight='bold')
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Test Accuracy")
plt.xticks(k_values)
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "8_accuracy_vs_k.png"))
plt.show()

In [ ]:
# Plot 9: Cross-Validation Accuracy
plt.figure(figsize=(9, 5))
cv_df = table5[table5['Fold'] != 'Average'].copy()
x_indices = np.arange(len(cv_df))
width = 0.35
plt.bar(x_indices - width/2, cv_df['Naïve Bayes'], width, label='Naïve Bayes (Gaussian)', color='steelblue')
plt.bar(x_indices + width/2, cv_df['Best KNN'], width, label='Best KNN', color='darkorange')
plt.axhline(y=nb_scores.mean(), color='blue', linestyle='--', label=f'NB Avg ({nb_scores.mean():.4f})')
plt.axhline(y=knn_scores.mean(), color='orange', linestyle='--', label=f'KNN Avg ({knn_scores.mean():.4f})')
plt.title("Plot 9: 5-Fold Cross-Validation Accuracy", fontsize=12, fontweight='bold')
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.xticks(x_indices, [f"Fold {f}" for f in cv_df['Fold']])
plt.ylim(0.7, 1.0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "9_cv_accuracy.png"))
plt.show()

In [ ]:
# Plot 10: Training Time Comparison
plt.figure(figsize=(8, 5))
models_list = list(exp_train_times.keys())
times_train_list = [exp_train_times[m] for m in models_list]
bars = plt.bar(models_list, times_train_list, color='teal')
plt.title("Plot 10: Training Time Comparison", fontsize=12, fontweight='bold')
plt.ylabel("Training Time (seconds)")
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, f"{yval:.6f}s", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "10_training_time_comparison.png"))
plt.show()

In [ ]:
# Plot 11: Prediction Time Comparison
plt.figure(figsize=(8, 5))
times_pred_list = [exp_pred_times[m] for m in models_list]
bars = plt.bar(models_list, times_pred_list, color='coral')
plt.title("Plot 11: Prediction Time Comparison", fontsize=12, fontweight='bold')
plt.ylabel("Prediction Time (seconds)")
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, f"{yval:.6f}s", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "11_prediction_time_comparison.png"))
plt.show()

In [ ]:
# Plot 12: Classifier Comparison Bar Chart
comp_rows = []
for name in all_preds.keys():
    pred = all_preds[name]
    comp_rows.append({
        'Classifier': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1-score': f1_score(y_test, pred, zero_division=0)
    })
comp_df = pd.DataFrame(comp_rows)
comp_melted = comp_df.melt(id_vars='Classifier', var_name='Metric', value_name='Score')

plt.figure(figsize=(10, 6))
sns.barplot(data=comp_melted, x='Classifier', y='Score', hue='Metric', palette='muted')
plt.title("Plot 12: Classifier Performance Comparison", fontsize=12, fontweight='bold')
plt.ylim(0.7, 1.0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "12_classifier_comparison_barchart.png"))
plt.show()

In [ ]:
# Plot 13: GridSearchCV Heatmap
plt.figure(figsize=(8, 6))
cv_results = pd.DataFrame(grid_search.cv_results_)
pivot_table = cv_results.pivot_table(index='param_n_neighbors', columns='param_metric', values='mean_test_score', aggfunc='max')
sns.heatmap(pivot_table, annot=True, fmt=".4f", cmap='YlGnBu', cbar_kws={'label': 'Mean CV Accuracy'})
plt.title("Plot 13: GridSearchCV Accuracy Heatmap (k vs Metric)", fontsize=12, fontweight='bold')
plt.xlabel("Distance Metric")
plt.ylabel("Number of Neighbors (k)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "13_gridsearch_heatmap.png"))
plt.show()

In [ ]:
# Plot 14: RandomizedSearchCV Score Distribution
plt.figure(figsize=(8, 5))
rand_scores = rand_search.cv_results_['mean_test_score']
sns.histplot(rand_scores, kde=True, color='seagreen', bins=10)
plt.axvline(rand_search.best_score_, color='red', linestyle='--', label=f"Best CV Score ({rand_search.best_score_:.4f})")
plt.title("Plot 14: RandomizedSearchCV Score Distribution", fontsize=12, fontweight='bold')
plt.xlabel("Mean CV Accuracy Score")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "14_randomizedsearch_score_distribution.png"))
plt.show()